In [5]:
import torch
from einops import einsum, rearrange
from jaxtyping import Float, Int
from torch import Tensor

from src.modules.attention import softmax

In [6]:
batch_size, seq_len, vocab_size = 4, 12, 100
torch.manual_seed(42)


logits = torch.randn(size=(batch_size, seq_len, vocab_size))
targets = torch.randint(low=0, high=vocab_size, size=(batch_size, seq_len))

In [15]:
predicted_soft = softmax(logits, dim=-1)

predicted_soft.shape, targets.shape

(torch.Size([4, 12, 100]), torch.Size([4, 12]))

In [17]:
predicted = torch.gather(logits, dim=-1, index=targets.unsqueeze(-1)).squeeze(
    -1
)

predicted.shape

torch.Size([4, 12])

In [21]:
predicted_2 = torch.take_along_dim(
    logits, targets.unsqueeze(-1), dim=-1
).squeeze(-1)

predicted_2.shape

torch.Size([4, 12])

In [26]:
max_logits = logits.max(-1, keepdim=True)[0]
norm_logits = logits - max_logits
predicted = torch.gather(norm_logits, -1, targets.unsqueeze(-1)).squeeze(-1)

sum_logits = torch.exp(norm_logits).sum(-1)

ce_loss = predicted - torch.log(sum_logits)

predicted.shape

torch.Size([4, 12])

In [22]:
torch.allclose(predicted, predicted_2)

True

In [ ]:
def cross_entropy_loss(
    logits: Float[Tensor, "... seq_len vocab_size"],
    targets: Float[Tensor, "... seq_len"],
):
    max_logits = logits.max(-1, keepdim=True)[0]
    norm_logits = logits - max_logits
    predicted = torch.gather(norm_logits, -1, targets.unsqueeze(-1)).squeeze(
        -1
    )

    sum_logits = torch.exp(norm_logits).sum(-1)

    ce_loss = predicted - torch.log(sum_logits)

    return ce_loss
